# Limpieza de Datos (Data Cleaning)

In [1]:
import pymysql
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import time
import datetime
from datetime import datetime

In [5]:
# Conectar a la base de datos
connection = pymysql.connect(
    host='212.227.90.6',
    user='Equipo17',
    password='E1q2u3i4p5o17',
    database='Equip_17' 
)

cursor = connection.cursor()
cursor.execute("SHOW TABLES")
tabla = [t[0] for t in cursor.fetchall()]

dataframes = {}

for nombre_tabla in tabla:
    df = pd.read_sql_query(f"SELECT * FROM {nombre_tabla}", connection)
    dataframes[nombre_tabla] = df
    print(f"Tabla {nombre_tabla} cargada exitosamente.")

C:\Users\Cristina\AppData\Local\Temp\ipykernel_3588\3229139825.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {nombre_tabla}", connection)


Tabla Tourist_Accommodation cargada exitosamente.


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7001 entries, 0 to 7000
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 7001 non-null   int64  
 1   name                         6998 non-null   object 
 2   description                  6972 non-null   object 
 3   host_id                      7001 non-null   int64  
 4   neighbourhood_name           7001 non-null   object 
 5   neighbourhood_district       4241 non-null   object 
 6   room_type                    7001 non-null   object 
 7   accommodates                 7001 non-null   int64  
 8   bathrooms                    6969 non-null   object 
 9   bedrooms                     6972 non-null   object 
 10  beds                         6998 non-null   float64
 11  amenities_list               6984 non-null   object 
 12  price                        6870 non-null   float64
 13  minimum_nights    

Antes de empezar a trabajar creamos una copia del DF donde realizaremos toda la limpieza y las transformaciones.

In [8]:
df_acomm = df.copy()

## Tratamiento de valores faltantes

In [30]:
df_acomm.isnull().sum()

apartment_id                      0
name                              3
description                      29
host_id                           0
neighbourhood_name                0
neighbourhood_district         2760
room_type                         0
accommodates                      0
bathrooms                        32
bedrooms                         29
beds                              3
amenities_list                   17
price                           131
minimum_nights                    0
maximum_nights                    0
has_availability                550
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              1254
last_review_date               1255
review_scores_rating           1327
review_scores_accuracy         1336
review_scores_cleanliness      1330
review_scores_checkin          1341
review_scores_communication 

Eliminamos la columna 'neighbourhood_district' ya que presenta muchos valores nulos y para este análisis no nos proporciona información relevante. 

In [42]:
df_acomm = df_acomm.drop(['neighbourhood_district'], axis='columns')

Verificamos cuantos registros presentan nulos en alguna de sus variables.

In [43]:
df_acomm[df_acomm.isnull().any(axis=1)]

,apartment_id,name,description,host_id,neighbourhood_name,room_type,accommodates,bathrooms,bedrooms,beds,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date
7,71603,PENTHOUSE1 BEST PRICE 15/21.07 PROMO LAST MINUTE!,The apartment you are about to book has everyt...,366654,la Dreta de l'Eixample,Entire home/apt,3,2,1,4.0,...,100.0,100.0,90.0,100.0,90.0,FALSO,42.0,spain,barcelona,2017-07-06
12,89756,"Old town 1, 6minutesfrom Ramblas",ONLY FOR FAMILIES or people over 25 years old!...,488117,el Raval,Entire home/apt,9,1,3,8.0,...,90.0,100.0,100.0,90.0,80.0,VERDADERO,17.0,spain,barcelona,2017-04-08
16,101580,Big house in PARADISE,Luxury and quiet villa in Cala Canyelles (2km ...,532063,Lloret de Mar,Entire home/apt,16,4,4,12.0,...,100.0,100.0,100.0,90.0,100.0,VERDADERO,73.0,spain,girona,2018-12-22
24,157327,House in Llofriu (Costa Brava),New rebuilt and furnished house pool bbq If yo...,755634,Forallac,Entire home/apt,8,5,4,8.0,...,100.0,100.0,100.0,80.0,80.0,FALSO,2.0,spain,girona,2020-04-30
29,176827,Se alquilan habitaciones en Madrid,Es un piso totalmente reformado. Todas las hab...,845399,Universidad,Private room,7,5,1,7.0,...,90.0,90.0,90.0,90.0,90.0,VERDADERO,248.0,spain,madrid,2017-04-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6991,27224489,Villa-Sierra,Bonito chalet de 3 plantas con jard�n y piscin...,203022185,Palafrugell,Entire home/apt,9,2,4,8.0,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,girona,2018-12-22
6992,27224529,Apartaments Sant Roc 4,Pleasant apartment second line from the sea an...,203022185,Palafrugell,Entire home/apt,6,1,2,4.0,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,girona,2019-03-30
6993,27224936,Apartaments Panoramic Arago-5,Charming apartment with community swimming poo...,203022185,Palafrugell,Entire home/apt,6,1,2,4.0,...,100.0,80.0,100.0,100.0,80.0,VERDADERO,23.0,spain,girona,2021-02-27
6997,27244243,101.108_New building apartment with two double...,Apartment in Cadaqu�s center. 1rst �floor. Ele...,151496825,Cadaqu�s,Entire home/apt,4,1,2,2.0,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,girona,2018-08-30


Comprobamos los datos de los registros que tienen nulos en la variable 'name'

In [45]:
df_acomm[df_acomm[['name']].isnull().any(axis=1)]

,apartment_id,name,description,host_id,neighbourhood_name,room_type,accommodates,bathrooms,bedrooms,beds,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date
1507,7164589,None,"exterior, bright, and charming room, in the ce...",37525983,Palacio,Private room,1,2,1,1.0,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,madrid,2020-10-17
1581,7576999,None,"This fantastic private bedroom, located in a p...",14415017,el Putxet i el Farr�,Private room,2,2,1,1.0,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,barcelona,2020-09-12
2165,11687495,None,What everyone loves most about my place is the...,48387429,Simancas,Entire home/apt,4,1,1,1.0,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,madrid,2019-06-07


Rellenamos los registros sin datos en la columna 'name' con una combinación entre 'room_type' y 'neighbourhood_name'

In [47]:
df_acomm['name'] = df_acomm['name'].fillna(df_acomm['room_type'] + ' ' + df_acomm['neighbourhood_name'])

Comprobamos los datos existentes en los registros que presentan nulos en la columna 'description'.

In [46]:
df_acomm[df_acomm[['description']].isnull().any(axis=1)]

,apartment_id,name,description,host_id,neighbourhood_name,room_type,accommodates,bathrooms,bedrooms,beds,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date
520,1921506,Apartments in Lloret de Mar 6pers.,None,8510344,Lloret de Mar,Entire home/apt,6,1,2,5.0,...,100.0,70.0,80.0,90.0,90.0,FALSO,18.0,spain,girona,2018-04-21
953,4150857,"Room @Eixample, 8 minutes walk from Passeig Gr...",None,4614901,la Dreta de l'Eixample,Private room,2,1,1,1.0,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,barcelona,2019-02-06
3751,17493843,Nice room in Gr�cia. 15 min. walking from center,None,30232813,la Vila de Gr�cia,Private room,1,1,1,1.0,...,100.0,90.0,90.0,100.0,100.0,FALSO,358.0,spain,barcelona,2019-07-10
5355,21774920,Piso c�ntrico con vistas,None,158657945,Quintana,Entire home/apt,5,1,2,4.0,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,madrid,2019-05-14
5795,23124779,Sitio muy agradable,None,165976120,Gaztambide,Private room,1,1,1,1.0,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,madrid,2019-02-06
6235,24231755,Villa Falco II,None,97086568,Calvi�,Entire home/apt,8,4,4,4.0,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,mallorca,2020-03-22
6440,24821982,APARTMENTO 4 BEDROOMS-WIFI,None,176073966,Embajadores,Entire home/apt,5,2,4,5.0,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,madrid,2020-09-13
6496,25025564,Apartamento Coqueto y Luminoso,None,189183311,Cruz De Humilladero,Entire home/apt,2,1,2,2.0,...,90.0,100.0,100.0,90.0,90.0,FALSO,25.0,spain,malaga,2019-09-01
6690,25929940,Duplex en Blasco Ib��ez 59 puerta 7 planta 2,None,194900549,L'AMISTAT,Entire home/apt,6,2,3,6.0,...,100.0,100.0,100.0,100.0,100.0,VERDADERO,24.0,spain,valencia,2019-03-31
6732,26133785,"Stylish, charming & cozy apartment in the hear...",None,2311093,les Tres Torres,Private room,1,1,1,2.0,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,barcelona,2019-07-10


Rellenamos los datos de la columna 'description' con los valores de la columna 'name'. 

In [49]:
df_acomm['description'] = df_acomm['description'].fillna(df_acomm['name'])

In [51]:
df_acomm.isnull().sum()

apartment_id                      0
name                              0
description                       0
host_id                           0
neighbourhood_name                0
room_type                         0
accommodates                      0
bathrooms                        32
bedrooms                         29
beds                              3
amenities_list                   17
price                           131
minimum_nights                    0
maximum_nights                    0
has_availability                550
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              1254
last_review_date               1255
review_scores_rating           1327
review_scores_accuracy         1336
review_scores_cleanliness      1330
review_scores_checkin          1341
review_scores_communication    1332
review_scores_location      

## Eliminación o corrección de duplicados

## Validación y corrección de valores atípicos

## Estandarización de formatos

## Corrección de tipos de datos

Modificamos los datos que presentan un formato erróneo. 

In [28]:
df_acomm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7001 entries, 0 to 7000
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7001 non-null   int64         
 1   name                         6998 non-null   object        
 2   description                  6972 non-null   object        
 3   host_id                      7001 non-null   int64         
 4   neighbourhood_name           7001 non-null   object        
 5   neighbourhood_district       4241 non-null   object        
 6   room_type                    7001 non-null   object        
 7   accommodates                 7001 non-null   int64         
 8   bathrooms                    6969 non-null   object        
 9   bedrooms                     6972 non-null   object        
 10  beds                         6998 non-null   float64       
 11  amenities_list               6984 non-null 

Los datos relativos a fecha que aparecían en formato alfanumérico.

In [ ]:
df_acomm['first_review_date'] = pd.to_datetime(df_acomm['first_review_date'], errors='coerce', format='%d/%m/%Y')

In [ ]:
df_acomm['last_review_date'] = pd.to_datetime(df_acomm['last_review_date'], errors='coerce', format='%d/%m/%Y')

In [ ]:
df_acomm['insert_date'] = pd.to_datetime(df_acomm['insert_date'], errors='coerce', format='%d/%m/%Y')

Las variables 'bedrooms' y 'bathrooms' aparecen como varchar y nos serán más útiles como INT.